In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
ROOT = ROOT.parents[1]


carpeta = ROOT / "data" / "raw" / "education" / "alumnes_estrangers_per_aula"

fitxers = {
    "2015-2016": carpeta / "curs_15_16.csv",
    "2016-2017": carpeta / "curs_16_17.csv",
    "2017-2018": carpeta / "curs_17_18.csv"
}
# Comprovem les columnes H (7), N (13) i P (15) amb el primer arxiu
df_prova = pd.read_csv(fitxers["2015-2016"], sep=None, engine="python", encoding="utf-8")
print("Columna H (índex 7):", df_prova.columns[7])
print("Columna N (índex 13):", df_prova.columns[13])
print("Columna P (índex 15):", df_prova.columns[15])
display(df_prova.iloc[:5, [7, 13, 15]])

Columna H (índex 7): Nivell
Columna N (índex 13): Zona nacionalitat (agrupació)
Columna P (índex 15): N Alumnes matriculats


,Nivell,Zona nacionalitat (agrupació),N Alumnes matriculats
0,1,CENTRE I SUDAMÈRICA,2
1,1,CENTRE I SUDAMÈRICA,1
2,1,ESPANYA,110
3,1,RESTA D'ÀFRICA,1
4,1,RESTA UNIÓ EUROPEA,1


In [2]:
resultats = []

for curs, ruta in fitxers.items():
    df = pd.read_csv(ruta, sep=None, engine="python", encoding="utf-8")
    
    # Extraiem les tres columnes per posició (H=7, N=13, P=15)
    nivell = df.iloc[:, 7].astype(str).str.strip()
    zona_nacional = df.iloc[:, 13].astype(str).str.strip().str.upper()
    alumnes = pd.to_numeric(df.iloc[:, 15], errors="coerce").fillna(0)
    
    # Condició comuna: zona nacional diferent d'ESPANYA (i que no sigui cel·la buida)
    es_estranger = (zona_nacional != "ESPANYA") & (zona_nacional != "NAN") & (zona_nacional != "")
    
    # 1. Alumnes estrangers a 4t d'ESO (nivell == '4')
    estrangers_4t = alumnes[es_estranger & (nivell == "4")].sum()
    
    # 2. Alumnes estrangers a tota l'ESO (nivells '1', '2', '3', '4')
    estrangers_eso_total = alumnes[es_estranger & (nivell.isin(["1", "2", "3", "4"]))].sum()
    
    # 3. Percentatge que representa 4t d'ESO respecte al total de l'etapa
    ratio_4t = estrangers_4t / estrangers_eso_total if estrangers_eso_total > 0 else 0
    
    resultats.append({
        "Curs": curs,
        "Estrangers_4t_ESO": int(estrangers_4t),
        "Estrangers_Total_ESO": int(estrangers_eso_total),
        "Ratio_4t_ESO": round(ratio_4t, 4)
    })

df_resum = pd.DataFrame(resultats)
display(df_resum)

print(f"\nFactor mitjà empíric (r_4t): {df_resum['Ratio_4t_ESO'].mean():.4f}")

,Curs,Estrangers_4t_ESO,Estrangers_Total_ESO,Ratio_4t_ESO
0,2015-2016,9347,36147,0.2586
1,2016-2017,8939,34922,0.2560
2,2017-2018,9148,36204,0.2527



Factor mitjà empíric (r_4t): 0.2558
